# 01 — eFeature Extraction

Build an `EModelEFeatureExtractionScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelEFeatureExtractionTask` for each coordinate. The task replicates the
`configure_targets()` + `pipeline.extract_efeatures()` flow from
[BluePyEModel/examples/L5PC/pipeline.py](https://github.com/openbraininstitute/BluePyEModel/blob/main/examples/L5PC/pipeline.py).

**Reads from:** `obi-output/L_emodel_optimization_inputs/` (created by `00_setup_download_l5pc_data.ipynb`).

**Writes to:** `obi-output/01_efeature_extraction/grid_scan/0/` — a self-contained
BluePyEModel working directory containing the copied inputs plus the new
`config/extract_config/L5PC_config.json`, `config/features/L5PC.json`,
and `figures/L5PC/extraction/` produced by feature extraction.


## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._01_efeature_extraction.blocks import (
    ExtractionInitialize,
)


## Resolve input paths

In [ ]:
inputs_root = (
    Path.cwd() / "../../../obi-output/L_emodel_optimization_inputs"
).resolve()
assert inputs_root.exists(), (
    f"{inputs_root} not found — run 00_setup_download_l5pc_data.ipynb first."
)
print("Inputs:", inputs_root)


## Build the scan config

Every `pipeline_settings` key relevant to extraction is exposed via blocks; defaults
match the L5PC recipe (Threshold=-20, interp_step=0.025, default_std_value=0.01, …).
The `targets` block defaults to the same protocols + features as `examples/L5PC/targets.py`.


In [ ]:
scan_config = obi.EModelEFeatureExtractionScanConfig(
    initialize=ExtractionInitialize(
        ephys_data_path=inputs_root / "ephys_data" / "C060109A1-SR-C1",
        morphology_path=inputs_root / "morphologies",
        mechanisms_path=inputs_root / "mechanisms",
        params_path=inputs_root / "config" / "params" / "pyr.json",
        recipes_path=inputs_root / "config" / "recipes.json",
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
)
print(scan_config.extraction_settings.to_dict(scan_config.efel_settings))


## Run the grid scan

Per-coordinate output goes to `obi-output/01_efeature_extraction/grid_scan/<idx>/`
(the `ZERO_INDEX` directory option keeps the path stable for downstream notebooks).


In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_efeature_extraction/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


## Inspect the extracted features

In [ ]:
import json

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)
print()

features_path = coord_root / "config" / "features" / "L5PC.json"
print("Features:", features_path)
features = json.loads(features_path.read_text())
print("Top-level keys:", list(features))
